In [3]:
import numpy as np
import matplotlib.pyplot as plt


def impute_row(row, mean, cov_inv):
    # create filters to extract rows and columns
    filv = np.nonzero(np.isnan(row))[0]
    film = np.ix_(filv, filv)

    # create the imputed values
    A = cov_inv
    x_c = (row - mean)
    x_c[np.isnan(x_c)] = 0
    imp =  mean[filv] - np.linalg.pinv(A[film]) @ (A @ x_c)[filv]
    row[np.isnan(row)] = imp

    # fix "music key" since it must be an integer from 0 to 11
    row[6] = np.round(row[6])

test_data = np.genfromtxt('test.csv',
                          delimiter=',',
                          missing_values='',
                          filling_values=np.nan,
                          skip_header=1)

train_data = np.genfromtxt('train.csv',
                          delimiter=',',
                          missing_values='',
                          filling_values=np.nan,
                          skip_header=1)
train_lbl = train_data[:, 14]
train_data = train_data[:, 0:14]

test_data_imp = test_data.copy()
train_data_imp = train_data.copy()

# filter out rows with missing values
test_fil = test_data[np.isfinite(np.sum(test_data, axis=1))]
train_fil = train_data[np.isfinite(np.sum(train_data, axis=1))]

# find mean and covariance (todo: can use the filtered rows?)
train_fil_mean = np.mean(train_fil, axis=0)
train_fil_cov = np.cov(train_fil, rowvar=False)
train_fil_cov_inv = np.linalg.pinv(train_fil_cov)

# now fill in the missing data
for i in range(len(train_data)):
    impute_row(train_data_imp[i], train_fil_mean, train_fil_cov_inv)

for i in range(len(test_data)):
    impute_row(test_data_imp[i], train_fil_mean, train_fil_cov_inv)


In [33]:
from catboost import CatBoostClassifier
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from skorch import NeuralNetBinaryClassifier
import torch
import torch.nn as nn

# Best parameters found:  {'colsample_bytree': np.float64(0.7068113003275547), 
# 'gamma': np.float64(0.20873898486652576), 'learning_rate': np.float64(0.01608374415773653),
#  'max_depth': 3, 'min_child_weight': 7, 'n_estimators': 498, 'reg_alpha': np.float64(1.5301963746469178),
#  'reg_lambda': np.float64(1.9359029489158148), 'subsample': np.float64(0.8309137610399533)}

torch.set_default_dtype(torch.float64)
N_SPLITS = 5  # Using 5 folds is a common and robust choice
kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# 2. Create a list to store the score from each fold
xgb_scores, rf_scores, avg_scores = [], [], []

print(f"Starting cross-validation for the model...")

print(train_lbl)
for fold, (train_index, val_index) in enumerate(kf.split(train_data_imp, train_lbl)):
    print(f"--- Fold {fold+1}/{N_SPLITS} ---")

    # Split the data for this fold
    X_train, X_val = train_data_imp[train_index], train_data_imp[val_index]
    y_train, y_val = train_lbl[train_index], train_lbl[val_index]
    
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    
    xgb_model = xgb.XGBClassifier(
        n_estimators=498,
        max_depth=3,
        learning_rate=0.017,
        subsample=0.8309137610399533,
        colsample_bytree=0.7068113003275547,
        gamma=0.20873898486652576,
        min_child_weight=7,
        reg_alpha=1.5301963746469178,
        reg_lambda=1.9359029489158148,
        objective='binary:logistic',  # Standard objective for binary classification
        eval_metric='logloss',          # A common evaluation metric for binary classification
        random_state=42,               # For reproducibility
        scale_pos_weight=0.57331
        # TODO: try with correct value
    )
    rf_model = CatBoostClassifier(
        bagging_temperature=0.9656320330745594,
        boosting_type='Plain',
        bootstrap_type='Bayesian',
        border_count=40,
        depth=5,
        iterations=152,
        l2_leaf_reg=3.078044430599341,
        learning_rate=0.077,
        min_data_in_leaf=7,
        random_strength=1.2199933155652418,
        rsm=1.0,
        scale_pos_weight= 0.57331, # 0.7600469802616581,
        random_state=42,
        verbose=False
    )

    # Train the model
    xgb_model.fit(X_train, y_train)
    rf_model.fit(X_train, y_train)

    # Make predictions on the validation set for this fold
    xgb_preds = xgb_model.predict_proba(X_val)[:, 1]
    rf_preds = rf_model.predict_proba(X_val)[:, 1]
    avg_preds = 0.2*xgb_preds + 0.8*rf_preds

    # Calculate and store the AUC score
    xgb_auc = roc_auc_score(y_val, xgb_preds)
    rf_auc = roc_auc_score(y_val, rf_preds)
    avg_auc = roc_auc_score(y_val, avg_preds)

    xgb_scores.append(xgb_auc)
    rf_scores.append(rf_auc)
    avg_scores.append(avg_auc)

    print(f"Fold {fold+1} XGB AUC: {xgb_auc:.5f}, RF AUC: {rf_auc:.5f}, AVG AUC: {avg_auc:.05f}")


# 4. Calculate and print the final average score
mean_xgb_auc = np.mean(xgb_scores)
mean_rf_auc = np.mean(rf_scores)
mean_avg_auc = np.mean(avg_scores)
std_xgb_auc = np.std(xgb_scores)
std_rf_auc = np.std(rf_scores)
std_avg_auc = np.std(avg_scores)

print("\n" + "="*40)
print("Cross-Validation Summary")
print(f"Mean XGB AUC Score: {mean_xgb_auc:.5f}")
print(f"Mean RF AUC Score: {mean_rf_auc:.5f}")
print(f"Mean AVG AUC Score: {mean_avg_auc:.5f}")

print(f"XGB Standard Deviation: {std_xgb_auc:.5f}")
print(f"RF Standard Deviation: {std_rf_auc:.5f}")
print(f"AVG Standard Deviation: {std_avg_auc:.5f}")
print("="*40)
"""
xgb_model.fit(train_data, train_lbl)

probs = xgb_model.predict_proba(test_data)
ans = np.stack([np.arange(0, len(test_data)), probs[:, 1]]).T
print(ans)
np.savetxt('submission.csv', ans, delimiter=',', header="id,song_popularity", comments="", fmt=["%.0f", "%.8f"])
"""

Starting cross-validation for the model...
[0. 1. 0. ... 1. 1. 1.]
--- Fold 1/5 ---
Fold 1 XGB AUC: 0.56337, RF AUC: 0.55969, AVG AUC: 0.56113
--- Fold 2/5 ---
Fold 2 XGB AUC: 0.57863, RF AUC: 0.57657, AVG AUC: 0.57788
--- Fold 3/5 ---
Fold 3 XGB AUC: 0.58847, RF AUC: 0.59011, AVG AUC: 0.59093
--- Fold 4/5 ---
Fold 4 XGB AUC: 0.57397, RF AUC: 0.57873, AVG AUC: 0.57853
--- Fold 5/5 ---
Fold 5 XGB AUC: 0.56628, RF AUC: 0.57019, AVG AUC: 0.56997

Cross-Validation Summary
Mean XGB AUC Score: 0.57414
Mean RF AUC Score: 0.57506
Mean AVG AUC Score: 0.57569
XGB Standard Deviation: 0.00898
RF Standard Deviation: 0.01002
AVG Standard Deviation: 0.00990


'\nxgb_model.fit(train_data, train_lbl)\n\nprobs = xgb_model.predict_proba(test_data)\nans = np.stack([np.arange(0, len(test_data)), probs[:, 1]]).T\nprint(ans)\nnp.savetxt(\'submission.csv\', ans, delimiter=\',\', header="id,song_popularity", comments="", fmt=["%.0f", "%.8f"])\n'

In [13]:
from sklearn.ensemble import RandomForestClassifier
import sklearn.metrics as met

rf_classifier = RandomForestClassifier(
    n_estimators=171,
    criterion='gini',
    max_depth=7,
    min_samples_split=12,
    min_samples_leaf=15,
    max_features='sqrt',
    class_weight='balanced_subsample',
    bootstrap=True,
    n_jobs=-1
)
rf_classifier.fit(train_data, train_lbl)

probs2 = rf_classifier.predict_proba(test_data)


In [15]:
ans2 = np.stack([np.arange(0, len(test_data)), probs2[:, 1]]).T
print(ans2)
np.savetxt('submission2.csv', ans2, delimiter=',', header="id,song_popularity", comments="", fmt=["%.0f", "%.8f"])

[[0.00000000e+00 4.94743310e-01]
 [1.00000000e+00 4.42226919e-01]
 [2.00000000e+00 4.45988644e-01]
 ...
 [9.99700000e+03 5.03918572e-01]
 [9.99800000e+03 4.78385219e-01]
 [9.99900000e+03 4.23514056e-01]]


In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

# 1. Define your base models (make sure they are not yet trained)
# Use your best parameters for each
estimators = [
    ('rf', RandomForestClassifier(
            n_estimators=171,
            criterion='gini',
            max_depth=7,
            min_samples_split=12,
            min_samples_leaf=15,
            max_features='sqrt',
            class_weight='balanced_subsample',
            bootstrap=True,
            n_jobs=-1
    )),
    ('xgb',xgb.XGBClassifier(
            n_estimators=443,
            max_depth=3,
            learning_rate=0.01469092202235818,
            subsample=0.8184644554526709,
            colsample_bytree=0.7888820517112247,
            gamma=0.08263346953150125,
            objective='binary:logistic',  # Standard objective for binary classification
            eval_metric='logloss',          # A common evaluation metric for binary classification
            random_state=42               # For reproducibility
    ))
]

# 2. Define the meta-model that will combine the predictions
# LogisticRegression is a common and effective choice
meta_model = LogisticRegression()

# 3. Create the Stacking Classifier
# cv=5 means it will use 5-fold cross-validation to generate the predictions
# that the meta-model is trained on.
stacking_model = StackingClassifier(
    estimators=estimators,
    final_estimator=meta_model,
    cv=5,
    verbose=2,
    passthrough=True
)

# 4. Train the entire stack
stacking_model.fit(train_data, train_lbl)

# 5. Get final predictions
probs3 = stacking_model.predict_proba(test_data)[:, 1]

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    5.3s finished
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.9s finished
/home/aditya/.pyenv/versions/ml/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [23]:
ans3 = np.stack([np.arange(0, len(test_data)), probs3]).T
print(ans3)
np.savetxt('submission3.csv', ans3, delimiter=',', header="id,song_popularity", comments="", fmt=["%.0f", "%.8f"])

[[0.00000000e+00 3.90121367e-01]
 [1.00000000e+00 3.94926064e-01]
 [2.00000000e+00 3.83921122e-01]
 ...
 [9.99700000e+03 3.82976522e-01]
 [9.99800000e+03 3.77851283e-01]
 [9.99900000e+03 3.97718042e-01]]


In [ ]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
import joblib

# Assume 'X_train' is your feature data and 'y_train' is the popularity (0 or 1)
# X_train = ...
# y_train = ...

# 1. Define the XGBoost Classifier model
xgb = XGBClassifier(objective='binary:logistic', eval_metric='logloss', scale_pos_weight=0.57331, device='cuda')

# 2. Define the hyperparameter search space
# Using distributions for a more flexible random search
param_dist = {
    'max_depth': randint(3, 12),
    'min_child_weight': randint(1, 10),
    'learning_rate': uniform(0.01, 0.29),
    'n_estimators': randint(100, 600),
    'gamma': uniform(0, 0.5),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'reg_alpha': uniform(0, 2),
    'reg_lambda': uniform(0.5, 4.5),
}

# IMPORTANT: If your classes are imbalanced, calculate and add 'scale_pos_weight'
# For example:
# count_neg = (y_train == 0).sum()
# count_pos = (y_train == 1).sum()
# xgb.set_params(scale_pos_weight=count_neg / count_pos)


# 3. Set up the Randomized Search with Cross-Validation
# n_iter: number of parameter settings that are sampled (e.g., 50)
# cv: number of folds in cross-validation (e.g., 5)
# n_jobs=-1: use all available CPU cores
random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=100,
    cv=5,
    scoring='roc_auc', # A good metric for binary classification
    verbose=2,
    n_jobs=4,
    return_train_score=True
)

# 4. Run the search
with joblib.parallel_backend('loky'):
    random_search.fit(train_data_imp, train_lbl)

# 5. Print the best parameters and best score
print("Best parameters found: ", random_search.best_params_)
print("Best ROC AUC score found: ", random_search.best_score_)

Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV] END colsample_bytree=0.7214579458670263, gamma=0.026080177696354068, learning_rate=0.09365948471270816, max_depth=9, min_child_weight=6, n_estimators=585, reg_alpha=1.8109108998929537, reg_lambda=3.1439653412949933, subsample=0.8945948045676011; total time=   4.0s
[CV] END colsample_bytree=0.7214579458670263, gamma=0.026080177696354068, learning_rate=0.09365948471270816, max_depth=9, min_child_weight=6, n_estimators=585, reg_alpha=1.8109108998929537, reg_lambda=3.1439653412949933, subsample=0.8945948045676011; total time=   4.0s
[CV] END colsample_bytree=0.7214579458670263, gamma=0.026080177696354068, learning_rate=0.09365948471270816, max_depth=9, min_child_weight=6, n_estimators=585, reg_alpha=1.8109108998929537, reg_lambda=3.1439653412949933, subsample=0.8945948045676011; total time=   4.2s
[CV] END colsample_bytree=0.7214579458670263, gamma=0.026080177696354068, learning_rate=0.09365948471270816, max_depth=9, min_c

In [54]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
import joblib
import numpy as np
from catboost import CatBoostClassifier

# Calculate scale_pos_weight for class imbalance
# Label 1 is 0.57331 times as frequent as label 0
scale_pos_weight = 1 / 0.57331  # ≈ 1.744

# Define parameter distributions for randomized search
param_distributions = {
    # Tree structure parameters
    'depth': randint(4, 11),
    'min_data_in_leaf': randint(1, 50),
    
    # Learning parameters
    'learning_rate': uniform(0.01, 0.29),
    'iterations': randint(100, 600),
    
    # Regularization parameters
    'l2_leaf_reg': uniform(1, 9),
    'random_strength': uniform(0, 2),
    'bagging_temperature': uniform(0, 1),
    
    # Sampling parameters
    'rsm': uniform(0.6, 0.4),  # Random subspace method (feature sampling)
    
    # Additional parameters
    'border_count': randint(32, 256),
    'boosting_type': ['Ordered', 'Plain'],
}

# Initialize CatBoost classifier with class imbalance handling
catboost_model = CatBoostClassifier(
    loss_function='Logloss',
    random_state=42,
    verbose=False,
    thread_count=1,
    bootstrap_type='Bayesian',  # Works well with bagging_temperature
    eval_metric='AUC',
    scale_pos_weight=0.57331,
)

# Initialize RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=catboost_model,
    param_distributions=param_distributions,
    n_iter=50,  # Number of parameter settings sampled
    scoring='roc_auc',
    cv=5,
    verbose=2,
    n_jobs=12,
    random_state=42,
    return_train_score=True
)

# 4. Run the search
with joblib.parallel_backend('loky'):
    random_search.fit(train_data_imp, train_lbl)

# 5. Print the best parameters and best score
print("Best parameters found: ", random_search.best_params_)
print("Best ROC AUC score found: ", random_search.best_score_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


/usr/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib/pyth

[CV] END bagging_temperature=0.3745401188473625, boosting_type=Ordered, border_count=46, depth=6, iterations=171, l2_leaf_reg=6.387926357773329, learning_rate=0.055245405728306586, min_data_in_leaf=19, random_strength=0.19994983163600577, rsm=0.7836995567863468; total time=   6.4s
[CV] END bagging_temperature=0.3745401188473625, boosting_type=Ordered, border_count=46, depth=6, iterations=171, l2_leaf_reg=6.387926357773329, learning_rate=0.055245405728306586, min_data_in_leaf=19, random_strength=0.19994983163600577, rsm=0.7836995567863468; total time=   6.5s
[CV] END bagging_temperature=0.3745401188473625, boosting_type=Ordered, border_count=46, depth=6, iterations=171, l2_leaf_reg=6.387926357773329, learning_rate=0.055245405728306586, min_data_in_leaf=19, random_strength=0.19994983163600577, rsm=0.7836995567863468; total time=   7.2s
[CV] END bagging_temperature=0.3745401188473625, boosting_type=Ordered, border_count=46, depth=6, iterations=171, l2_leaf_reg=6.387926357773329, learning_

In [35]:
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(train_data_imp)
X_test = scaler.transform(test_data_imp)
y_train = train_lbl

# Best parameters found:  {'bagging_temperature': np.float64(0.9656320330745594),
# 'boosting_type': 'Plain', 'border_count': 40, 'depth': 5, 'iterations': 152,
# 'l2_leaf_reg': np.float64(3.078044430599341), 'learning_rate': np.float64(0.07989738514754338),
# 'min_data_in_leaf': 7, 'random_strength': np.float64(1.2199933155652418),
# 'rsm': np.float64(0.9332779646944658), 'scale_pos_weight': np.float64(0.7600469802616581)}
xgb_models, cb_models = [], []

for i in range(0, 1):
    xgb_models.append(xgb.XGBClassifier(
        n_estimators=498,
        max_depth= 3,
        learning_rate=0.017,
        subsample=0.8309137610399533,
        colsample_bytree=0.7068113003275547,
        gamma=0.20873898486652576,
        min_child_weight=7,
        reg_alpha=1.5301963746469178,
        reg_lambda=1.9359029489158148,
        objective='binary:logistic',  # Standard objective for binary classification
        eval_metric='logloss',          # A common evaluation metric for binary classification
        scale_pos_weight=0.57331,
        random_state=42
        # TODO: try with correct value
    ))
    cb_models.append(CatBoostClassifier(
        bagging_temperature=0.9656320330745594,
        boosting_type='Plain',
        bootstrap_type='Bayesian',
        border_count=40,
        depth=5,
        iterations=152,
        l2_leaf_reg=3.078044430599341,
        learning_rate=0.077,
        min_data_in_leaf=7,
        random_strength=1.2199933155652418,
        rsm=1.0,
        scale_pos_weight= 0.57331, # 0.7600469802616581,
        random_state=42,
        verbose=False
    ))

# Train the model
for i in range(0, 1):
    print(f"{i}th iteration...")
    xgb_models[i].fit(X_train, y_train)
    cb_models[i].fit(X_train, y_train)

# Make predictions on the validation set for this fold
xgb_preds = np.zeros((len(X_test)))
for i in range(0, 1):
    xgb_preds += xgb_models[i].predict_proba(X_test)[:, 1]
#xgb_preds /= 20;

cb_preds = np.zeros((len(X_test)))
for i in range(0, 1):
    cb_preds += cb_models[i].predict_proba(X_test)[:, 1]
#cb_preds /= 20;

avg_preds = 0.2*xgb_preds + 0.8*cb_preds

ans4 = np.stack([np.arange(0, len(test_data)), avg_preds]).T
print(ans4)
np.savetxt('submission.csv', ans4, delimiter=',', header="id,song_popularity", comments="", fmt=["%.0f", "%.8f"])

0th iteration...
[[0.00000000e+00 2.46809149e-01]
 [1.00000000e+00 1.94763571e-01]
 [2.00000000e+00 2.32188651e-01]
 ...
 [9.99700000e+03 2.47470247e-01]
 [9.99800000e+03 2.52942561e-01]
 [9.99900000e+03 1.72316562e-01]]
